# NQ hourly pattern on Yahoo data

**Goal.** After the daily reversal screen failed, check whether NQ hourly bars revert after a large hour.

**Method.** Yahoo `NQ=F`, `period="730d"`, `interval="1h"` (this pull is July 2024 through November 2026). Z-score uses the last 100 hourly returns, shifted by one bar. The regular-hours split is 9:00–15:00 New York. The sample is also cut in half by time.

**Result.** After an RTH hour up more than 2 standard deviations: next hour down 57.3% of the time, average move -15.8 points, 260 bars. The two halves of the window agree in sign. Overnight bars do not. The four-hour follow-through is about flat. This is pattern-finding inside one window. It is not an out-of-sample test. Notebook 06 tries to trade it. Notebook 08 rebuilds it on minute data from 2017.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

from src.signals import return_zscore_series

nq = yf.download("NQ=F", period="730d", interval="1h", multi_level_index=False, progress=False)["Close"].dropna()
frame = return_zscore_series(nq, 100)
r, z = frame["r"], frame["z"]
next_1h = r.shift(-1)
next_4h = nq.shift(-4) / nq - 1
avg_price = nq.mean()

print(f"NQ 1-hour bars: {len(nq)} bars, {nq.index[0].date()} to {nq.index[-1].date()}\n")

for z_cut in [2.0, 2.5, 3.0]:
    for side, bars in [("UP", z > z_cut), ("DOWN", z < -z_cut)]:
        n1 = next_1h[bars].dropna()
        n4 = next_4h[bars].dropna()
        reverse = (n1 < 0).mean() if side == "UP" else (n1 > 0).mean()
        print(f"{side:4} hour > {z_cut}x vol: {bars.sum():4} bars | next hour reverses {reverse:.1%} "
              f"| avg next 1h {n1.mean() * avg_price:+6.1f} pts | avg next 4h {n4.mean() * avg_price:+6.1f} pts")
    print()


NQ 1-hour bars: 13723 bars, 2024-07-07 to 2026-11-25

UP   hour > 2.0x vol:  379 bars | next hour reverses 54.1% | avg next 1h  -10.1 pts | avg next 4h   +0.9 pts
DOWN hour > 2.0x vol:  447 bars | next hour reverses 48.5% | avg next 1h   -4.5 pts | avg next 4h   +5.5 pts

UP   hour > 2.5x vol:  227 bars | next hour reverses 54.6% | avg next 1h  -10.6 pts | avg next 4h   +8.2 pts
DOWN hour > 2.5x vol:  285 bars | next hour reverses 47.7% | avg next 1h   -8.7 pts | avg next 4h   +7.8 pts

UP   hour > 3.0x vol:  136 bars | next hour reverses 51.5% | avg next 1h   -9.2 pts | avg next 4h  +21.3 pts
DOWN hour > 3.0x vol:  189 bars | next hour reverses 49.2% | avg next 1h  -14.1 pts | avg next 4h   -0.5 pts



In [3]:
df_i = pd.DataFrame({"z": z, "n1": next_1h, "n4": next_4h}).dropna()
df_i["hour"] = df_i.index.tz_convert("America/New_York").hour
up = df_i[df_i["z"] > 2.0].copy()
up["session"] = np.where(up["hour"].between(9, 15), "RTH (9am-4pm)", "Overnight")
mid = df_i.index[0] + (df_i.index[-1] - df_i.index[0]) / 2
up["half"] = np.where(up.index < mid, "Year 1", "Year 2")

for col in ["session", "half"]:
    print(f"\nBy {col}:")
    for name, g in up.groupby(col):
        print(f"{name:14} {len(g):3} bars | next hour down {(g.n1 < 0).mean():.1%} "
              f"| avg next 1h {g.n1.mean() * avg_price:+6.1f} pts | avg next 4h {g.n4.mean() * avg_price:+6.1f} pts")



By session:
Overnight      118 bars | next hour down 47.5% | avg next 1h   +0.2 pts | avg next 4h   +4.2 pts
RTH (9am-4pm)  260 bars | next hour down 57.3% | avg next 1h  -15.8 pts | avg next 4h   -0.6 pts

By half:
Year 1         185 bars | next hour down 54.6% | avg next 1h   -8.8 pts | avg next 4h   +8.5 pts
Year 2         193 bars | next hour down 53.9% | avg next 1h  -12.7 pts | avg next 4h   -6.4 pts


RTH hour up more than 2x trailing volatility: next hour down 57.3%, average -15.8 points, 260 bars. Both halves of the sample point the same way. Overnight is noise. The four-hour move is noise.
